# A/B Testing for New Web Site

An e-commerce company wants to test a website with chatbot

- Version A (Control) → Old website without chatbot
- Version B (Treatment) → New website with chatbot

Goal:

 > Increase purchase conversion rate


### Each Step
```
Users visit website
        ↓
Randomly see website A or B
        ↓
Some users purchase
        ↓
Compute conversion rates
        ↓
Z-test checks significance
        ↓
Product decision


In [21]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

np.random.seed(42)

# Generate Experiment Data

In [22]:
n_users = 8000

# Old website A conversion rate
control_rate = 0.08     # 8%

# New website B conversion rate (slightly better)
treatment_rate = 0.095  # 9.5%

# Generate User Outcomes
# 1 = purchase
# 0 = no purchase

control = np.random.binomial(1, control_rate, n_users) # website A
treatment = np.random.binomial(1, treatment_rate, n_users) # website B

In [30]:
print("A (no chatbot)  :", control[:20]) # A
print("B (with chatbot):", treatment[:20]) # B

A (no chatbot)  : [0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
B (with chatbot): [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


# Calculate Conversions

In [31]:
samples = np.array([
    len(control),  # total samples for A
    len(treatment) # total samples for B
])

successes = np.array([
    control.sum(),     # success for A
    treatment.sum()    # success for B
])

print("samples  :", samples)
print("successes:", successes)

samples  : [8000 8000]
successes: [635 743]


In [24]:
conv_A = successes[0] / samples[0]
conv_B = successes[1] / samples[1]

print("Conversion Rate (Old website-no chatbat):", round(conv_A, 4))
print("Conversion Rate (New website-with chatbot):", round(conv_B, 4))

Conversion Rate (Old website-no chatbat): 0.0794
Conversion Rate (New website-with chatbot): 0.0929


# Define Hypotheses
Let
- Prob_A = probability a user purchases using OLD website
- Prob_B = probability a user purchases using NEW website

**Null Hypotheses**: No change. Probability or chance of user purchasing is same with or without the chatbot

**Alternative Hypotheses**: There is improvement. Probability or chance of user purchasing is more with chatbot

In [26]:
print("\nH0: No change. (New website performs the same):  Prob_B = Prob_A")
print("H1: (New website improves conversions): Prob_B > Prob_A")


H0: No change. (New website performs the same):  Prob_B = Prob_A
H1: (New website improves conversions): Prob_B > Prob_A


# Proportion Z-Test

### Why do we use Proportion Z-Test
> Because we are comparing two probabilities (proportions) coming from binary outcomes(0/1) with large sample sizes.

We want to know: Is the purchase probability different between two groups?
That is 
```
Prob_A = purchase probability with old website
versus
Prob_B = purchase probability with new website

So we must compare two proportions. 
Hence, we use Proportion Z-test.
```

In [27]:
z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=samples,
    alternative='larger'   # testing improvement
)

print("\nZ-statistic:", round(z_stat, 4))
print("P-value:", round(p_value, 6))


Z-statistic: -3.0434
P-value: 0.99883


# Decision Rule

In [28]:
alpha = 0.05

print("\nDecision:")

if p_value < alpha:
    print("Reject H0 — Launch NEW website with chatbot design")
else:
    print(" Fail to Reject H0 — Keep OLD website")


Decision:
 Fail to Reject H0 — Keep OLD website
